# Dịch máy Anh-Việt: BPE + seq2seq attention + beam search

Bài portfolio: một hệ dịch máy thần kinh hoàn chỉnh train trên 133 317 cặp câu IWSLT'15, đánh giá bằng BLEU và chrF của
sacrebleu trên `tst2013`, có ablation và phân tích lỗi. Thiết kế là của bạn; notebook chỉ gợi ý bố cục.

**Cách làm bài**
- Chỉ một hàm được chấm bằng test: `beam_search` (cell dưới, đặc tả trong docstring). Phần còn lại tự thiết kế.
- Notebook phải ghi bản dịch `tst2013` của model tốt nhất vào `tst2013.hyp.vi` (cùng thư mục với notebook, 1 268 dòng);
  test chấm BLEU của tệp này.
- GPU: chạy notebook bằng kernel `ai-gpu` (xem `setup/gpu/README.md`). Env GPU không có `sacrebleu`; chấm BLEU trong
  env gốc, hoặc chạy `uv run --project setup/gpu --with sacrebleu==2.6.0 ...`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/04-nlp-sequence-models/exercises/advanced/01-nmt-en-vi -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải IWSLT'15 nếu chưa có
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

IWSLT = ensure("iwslt15-en-vi")

In [ ]:
import math

import torch

## 1. Beam search (được chấm)

`beam_search(step_fn, bos, eos, beam_size, max_len, alpha=0.6) -> list[int]`:

- `step_fn(prefixes)` nhận danh sách tiền tố (mỗi cái bắt đầu bằng `bos`) và trả `Tensor (len(prefixes), V)` log-xác suất
  của token kế tiếp. Model của bạn bọc trong một hàm như vậy (giữ cache state theo tiền tố nếu muốn nhanh).
- Bắt đầu từ một beam `[bos]` điểm 0. Mỗi bước: mở rộng mọi beam còn sống với mọi token, điểm = tổng log-xác suất; xét
  các ứng viên theo điểm giảm dần: ứng viên kết thúc bằng `eos` vào danh sách **xong** (không mở rộng tiếp), còn lại lấy
  tối đa `beam_size` ứng viên tốt nhất làm beam sống của bước sau (chỉ xét `2 * beam_size` ứng viên đầu là đủ).
- Dừng khi đã có ít nhất `beam_size` câu xong, hoặc sau `max_len` bước. Không có câu nào xong thì dùng các beam sống.
- Chọn câu có điểm chuẩn hoá theo độ dài cao nhất (Wu và cộng sự, 2016):
  $$\text{score} / \left(\frac{5 + L}{6}\right)^{\alpha}$$ với $$L$$ là số token sinh ra (kể cả `eos`, không kể `bos`).
- Trả token sinh ra, **không** gồm `bos` và `eos`.

In [ ]:
def beam_search(step_fn, bos: int, eos: int, beam_size: int, max_len: int, alpha: float = 0.6) -> list[int]:
    # TODO
    raise NotImplementedError

## 2. Gợi ý bố cục (không chấm)

1. Dữ liệu: `html.unescape`, giữ chữ hoa (BLEU so với reference có chữ hoa), lọc cặp quá dài khi train.
2. Tokenizer: BPE riêng cho mỗi ngôn ngữ (bạn đã tự viết ở Basic 01; ở đây dùng `tokenizers` cho nhanh), vocab 8k.
3. Model: seq2seq của Intermediate 03, lớn hơn (embedding 256, hidden 512), dropout, nhiều epoch; batch theo độ dài.
4. Chọn checkpoint theo BLEU trên `tst2012` (dev), chỉ dùng `tst2013` một lần ở cuối.
5. Decode `tst2013` bằng greedy và beam, ghi `tst2013.hyp.vi`; BLEU, chrF, chữ ký sacrebleu.
6. Ablation, attention heatmap, đường cong học, phân tích lỗi theo độ dài câu. Viết `REPORT.md`.

In [ ]:
# explore: TODO: pipeline của bạn